# P3 · Customer Segmentation — RFM Scoring & Standard Segments

**Project:** P3 · Coffra Customer Segmentation
**Author:** Sebastian Kradyel
**Date:** April 2026
**Notebook:** 03_rfm_scoring_and_segments.ipynb

---

## Purpose

This notebook computes **RFM metrics** (Recency, Frequency, Monetary), scores customers on quintiles, and assigns standard segment labels following the established RFM segmentation framework. The output is a per-customer dataset ready for clustering analysis (next notebook) and strategy mapping (notebook 4).

## Methodology

**RFM definitions used:**
- **Recency (R):** Days since last purchase, measured from a fixed snapshot date (the day after the dataset's last transaction).
- **Frequency (F):** Number of unique invoices per customer.
- **Monetary (M):** Total revenue per customer in GBP.

**Scoring approach:** Quintile scoring (1–5 scale) on each dimension. Recency is reverse-coded (5 = most recent). Combined into an `RFM_Score` (e.g., "555") and mapped to 11 standard segments using the `R + FM` reduced framework, which is industry standard for non-technical stakeholders.

**Standard 11-segment framework:**
Champions, Loyal Customers, Potential Loyalists, Recent Customers, Promising, Customers Needing Attention, About to Sleep, At Risk, Cannot Lose Them, Hibernating, Lost.

## Why RFM (not direct ML clustering)

RFM is the right starting point because:
1. **Interpretability:** Marketing stakeholders understand "this customer hasn't bought in 6 months" instinctively. They do not understand "this customer is in cluster 3."
2. **Action-orientation:** Each segment maps to a known retention action.
3. **Foundation for clustering:** RFM scores serve as features for clustering in the next notebook, providing apples-to-apples comparison between rule-based and ML segmentation.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Brand-aligned styling
COFFRA_BROWN = '#3E2723'
COFFRA_BROWN_LIGHT = '#6D4C41'
COFFRA_CREAM = '#EFEBE9'
COFFRA_PALETTE = [COFFRA_BROWN, COFFRA_BROWN_LIGHT, '#A1887F', '#BCAAA4', '#D7CCC8', '#8D6E63']

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.titlesize'] = 13

INPUT_PATH = Path('../data/processed/online_retail_cleaned.parquet')
OUTPUT_DIR = Path('../data/processed')

## 2. Load Cleaned Data

In [ ]:
df = pd.read_parquet(INPUT_PATH)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f'Loaded: {df.shape[0]:,} transactions for {df["Customer ID"].nunique():,} customers')
print(f'Date range: {df.InvoiceDate.min().date()} → {df.InvoiceDate.max().date()}')
df.head()

## 3. Compute RFM Metrics

### Snapshot date choice

The snapshot date is the reference point for measuring recency. We use **the day after the last transaction** in the dataset, simulating an analyst running the report immediately after the data was extracted. This is the industry-standard convention.

For real production deployment, the snapshot would be **today's date**, recomputed at each report run.

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f'Snapshot date: {snapshot_date.date()}')
print(f'Most recent transaction in data: {df.InvoiceDate.max()}')

In [ ]:
# Compute per-customer RFM
rfm = df.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('LineRevenue', 'sum'),
).reset_index()

print(f'RFM table shape: {rfm.shape}')
print(f'Unique customers: {rfm["Customer ID"].nunique():,}')
print('\nRFM summary statistics:')
print(rfm[['Recency', 'Frequency', 'Monetary']].describe(percentiles=[0.05, 0.5, 0.95, 0.99]).round(2))

### Distribution of each RFM dimension

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].hist(rfm['Recency'], bins=50, color=COFFRA_BROWN, alpha=0.85)
axes[0].set_title('Recency Distribution\n(days since last purchase)')
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Customers')
axes[0].axvline(rfm['Recency'].median(), color='red', linestyle='--', label=f"Median: {rfm['Recency'].median():.0f}")
axes[0].legend()

axes[1].hist(rfm['Frequency'].clip(upper=30), bins=30, color=COFFRA_BROWN_LIGHT, alpha=0.85)
axes[1].set_title('Frequency Distribution\n(invoices, clipped at 30)')
axes[1].set_xlabel('Number of invoices')
axes[1].set_ylabel('Customers')
axes[1].set_yscale('log')
axes[1].axvline(rfm['Frequency'].median(), color='red', linestyle='--', label=f"Median: {rfm['Frequency'].median():.0f}")
axes[1].legend()

axes[2].hist(rfm['Monetary'].clip(upper=10000), bins=50, color='#A1887F', alpha=0.85)
axes[2].set_title('Monetary Distribution\n(£ total, clipped at £10K)')
axes[2].set_xlabel('Revenue (£)')
axes[2].set_ylabel('Customers')
axes[2].set_yscale('log')
axes[2].axvline(rfm['Monetary'].median(), color='red', linestyle='--', label=f"Median: £{rfm['Monetary'].median():.0f}")
axes[2].legend()

plt.tight_layout()
plt.show()

**Observations:**
- **Recency:** Bimodal — many recent customers (last 30–60 days) plus a long tail of inactive customers. Healthy mix for segmentation.
- **Frequency:** Heavily right-skewed. Most customers have purchased only a few times; a small group are frequent buyers. Typical retail pattern.
- **Monetary:** Long-tail distribution, classical Pareto. The top 5% of customers will dominate revenue.

## 4. Quintile Scoring

Each customer receives a score from 1 to 5 on each RFM dimension based on which quintile they fall into.

**Convention:**
- **R_Score:** Lower recency = higher score (5 = most recent).
- **F_Score:** Higher frequency = higher score.
- **M_Score:** Higher monetary = higher score.

We use `qcut` for quintile-based scoring with `duplicates='drop'` to handle ties at quintile boundaries.

In [ ]:
# Recency: lower is better, so reverse the labels
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1], duplicates='drop').astype(int)

# Frequency: higher is better. Use rank to break ties consistently for skewed distributions.
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Monetary: higher is better
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

# Combined FM score (for the simplified 11-segment framework)
rfm['FM_Score'] = ((rfm['F_Score'] + rfm['M_Score']) / 2).round().astype(int)

print('RFM scoring complete.')
print('Sample of scored customers:')
rfm.head(10)

In [ ]:
# Verify quintile distributions are roughly balanced
print('R_Score distribution:')
print(rfm['R_Score'].value_counts().sort_index())
print('\nF_Score distribution:')
print(rfm['F_Score'].value_counts().sort_index())
print('\nM_Score distribution:')
print(rfm['M_Score'].value_counts().sort_index())

## 5. Assign Standard 11-Segment Labels

We use the simplified 2D framework based on **R_Score** and **FM_Score**, which is the standard in industry for clarity.

In [ ]:
def assign_segment(row):
    r, fm = row['R_Score'], row['FM_Score']

    # Top tier — recent and high-value
    if r >= 5 and fm >= 5:
        return 'Champions'
    if r >= 4 and fm >= 4:
        return 'Loyal Customers'

    # New / promising customers
    if r >= 5 and fm == 1:
        return 'Recent Customers'
    if r >= 4 and fm <= 2:
        return 'Promising'
    if r >= 4 and fm >= 3:
        return 'Potential Loyalists'

    # Middle ground — needs attention
    if r == 3 and fm >= 3:
        return 'Customers Needing Attention'
    if r == 3 and fm <= 2:
        return 'About to Sleep'

    # At-risk customers
    if r <= 2 and fm >= 4:
        return 'Cannot Lose Them'
    if r <= 2 and fm == 3:
        return 'At Risk'

    # Lost / hibernating
    if r == 2 and fm <= 2:
        return 'Hibernating'
    if r == 1:
        return 'Lost'

    return 'Other'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

segment_summary = rfm['Segment'].value_counts().reset_index()
segment_summary.columns = ['Segment', 'Customers']
segment_summary['Pct_of_Total'] = (segment_summary['Customers'] / len(rfm) * 100).round(1)
print('Segment distribution:')
print(segment_summary)

### Segment-level economics

Beyond customer counts, what does each segment contribute to revenue?

In [ ]:
segment_econ = rfm.groupby('Segment').agg(
    customers=('Customer ID', 'count'),
    avg_recency=('Recency', 'mean'),
    avg_frequency=('Frequency', 'mean'),
    avg_monetary=('Monetary', 'mean'),
    total_monetary=('Monetary', 'sum'),
).round(2)

segment_econ['pct_customers'] = (segment_econ['customers'] / segment_econ['customers'].sum() * 100).round(1)
segment_econ['pct_revenue'] = (segment_econ['total_monetary'] / segment_econ['total_monetary'].sum() * 100).round(1)

# Order by revenue contribution
segment_econ = segment_econ.sort_values('total_monetary', ascending=False)
print('Segment economics (sorted by revenue):')
print(segment_econ[['customers', 'pct_customers', 'pct_revenue', 'avg_recency', 'avg_frequency', 'avg_monetary']])

In [ ]:
# Visualize: customer share vs revenue share per segment
segment_econ_sorted = segment_econ.sort_values('pct_revenue', ascending=True)

fig, ax = plt.subplots(figsize=(11, 6))
y = np.arange(len(segment_econ_sorted))
width = 0.4

ax.barh(y - width/2, segment_econ_sorted['pct_customers'], width, label='% of Customers', color=COFFRA_BROWN_LIGHT, alpha=0.85)
ax.barh(y + width/2, segment_econ_sorted['pct_revenue'], width, label='% of Revenue', color=COFFRA_BROWN, alpha=0.85)

ax.set_yticks(y)
ax.set_yticklabels(segment_econ_sorted.index)
ax.set_xlabel('Percentage')
ax.set_title('Customer Share vs. Revenue Share by Segment')
ax.legend()
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

### Pareto check — Top 20% revenue concentration

In [ ]:
rfm_sorted = rfm.sort_values('Monetary', ascending=False).reset_index(drop=True)
rfm_sorted['cumulative_revenue_pct'] = rfm_sorted['Monetary'].cumsum() / rfm_sorted['Monetary'].sum() * 100
rfm_sorted['cumulative_customers_pct'] = (rfm_sorted.index + 1) / len(rfm_sorted) * 100

# Find percentile where 80% revenue is reached
p80 = rfm_sorted[rfm_sorted['cumulative_revenue_pct'] >= 80].iloc[0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rfm_sorted['cumulative_customers_pct'], rfm_sorted['cumulative_revenue_pct'],
        color=COFFRA_BROWN, linewidth=2.2)
ax.fill_between(rfm_sorted['cumulative_customers_pct'], rfm_sorted['cumulative_revenue_pct'],
                alpha=0.15, color=COFFRA_BROWN)

ax.axvline(20, color='gray', linestyle=':', linewidth=1)
ax.axhline(80, color='gray', linestyle=':', linewidth=1)
ax.scatter([p80['cumulative_customers_pct']], [80], color='red', s=80, zorder=5)
ax.annotate(f"  Top {p80['cumulative_customers_pct']:.1f}% customers = 80% revenue",
            xy=(p80['cumulative_customers_pct'], 80), fontsize=10)

ax.set_xlabel('Cumulative % of Customers')
ax.set_ylabel('Cumulative % of Revenue')
ax.set_title('Pareto Curve — Customer Revenue Concentration')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Top 20% of customers contribute {rfm_sorted.iloc[int(len(rfm_sorted)*0.20)]["cumulative_revenue_pct"]:.1f}% of revenue.')

**Insight:** Strong Pareto pattern. The top ~20% of customers contribute ~70-80% of revenue, justifying differentiated retention investment for high-value segments.

## 6. Visual Heatmap — R × FM Cross-Tabulation

In [ ]:
# Customers heatmap: count per (R_Score, FM_Score) cell
heatmap_count = rfm.pivot_table(
    index='FM_Score', columns='R_Score', values='Customer ID', aggfunc='count'
).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(heatmap_count.iloc[::-1], annot=True, fmt=',', cmap='YlOrBr',
            cbar_kws={'label': 'Number of Customers'}, ax=ax,
            linewidths=0.5, linecolor='white')
ax.set_title('Customer Distribution: FM_Score (rows) × R_Score (columns)')
ax.set_xlabel('R_Score (Recency — higher = more recent)')
ax.set_ylabel('FM_Score (Frequency × Monetary)')
plt.tight_layout()
plt.show()

In [ ]:
# Revenue heatmap: total monetary value per cell
heatmap_revenue = rfm.pivot_table(
    index='FM_Score', columns='R_Score', values='Monetary', aggfunc='sum'
).fillna(0)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(heatmap_revenue.iloc[::-1], annot=True, fmt=',.0f', cmap='YlOrBr',
            cbar_kws={'label': 'Total Revenue (£)'}, ax=ax,
            linewidths=0.5, linecolor='white')
ax.set_title('Revenue Distribution: FM_Score (rows) × R_Score (columns)')
ax.set_xlabel('R_Score (Recency — higher = more recent)')
ax.set_ylabel('FM_Score (Frequency × Monetary)')
plt.tight_layout()
plt.show()

## 7. Segment Profiles

Compare segments side-by-side on RFM characteristics to validate segment definitions are meaningful.

In [ ]:
# Order segments from best to worst (Champions → Lost) for clarity
segment_order = [
    'Champions', 'Loyal Customers', 'Potential Loyalists',
    'Recent Customers', 'Promising',
    'Customers Needing Attention',
    'Cannot Lose Them', 'At Risk', 'About to Sleep',
    'Hibernating', 'Lost'
]
segment_order = [s for s in segment_order if s in rfm['Segment'].unique()]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Recency by segment
rfm_ordered = rfm[rfm['Segment'].isin(segment_order)].copy()
rfm_ordered['Segment'] = pd.Categorical(rfm_ordered['Segment'], categories=segment_order, ordered=True)

sns.boxplot(data=rfm_ordered, x='Recency', y='Segment', ax=axes[0], color=COFFRA_BROWN)
axes[0].set_title('Recency by Segment')
axes[0].set_xlim(0, 500)

sns.boxplot(data=rfm_ordered, x='Frequency', y='Segment', ax=axes[1], color=COFFRA_BROWN_LIGHT)
axes[1].set_title('Frequency by Segment')
axes[1].set_xlim(0, 30)
axes[1].set_ylabel('')

sns.boxplot(data=rfm_ordered, x='Monetary', y='Segment', ax=axes[2], color='#A1887F')
axes[2].set_title('Monetary by Segment')
axes[2].set_xlim(0, 5000)
axes[2].set_ylabel('')

plt.tight_layout()
plt.show()

**Validation:** Segments separate cleanly along expected dimensions. Champions and Loyal Customers cluster at low recency, high frequency, high monetary. Lost segment shows uniformly high recency and low engagement metrics. This confirms the segment definitions are meaningful and align with business intuition.

## 8. Save Outputs

Save the full RFM table for use in the next notebooks (clustering and strategy mapping).

In [ ]:
rfm_output_path = OUTPUT_DIR / 'rfm_scored.parquet'
rfm.to_parquet(rfm_output_path, index=False)
print(f'Saved: {rfm_output_path}')
print(f'Rows: {len(rfm):,}')
print(f'Size: {rfm_output_path.stat().st_size / 1024:.1f} KB')

# Save summary stats as JSON for dashboard consumption
import json
rfm_summary = {
    'snapshot_date': str(snapshot_date.date()),
    'total_customers': int(len(rfm)),
    'segments': {
        seg: {
            'customers': int(group['Customer ID'].count()),
            'pct_customers': round(group['Customer ID'].count() / len(rfm) * 100, 1),
            'avg_recency': round(float(group['Recency'].mean()), 1),
            'avg_frequency': round(float(group['Frequency'].mean()), 1),
            'avg_monetary': round(float(group['Monetary'].mean()), 2),
            'total_monetary': round(float(group['Monetary'].sum()), 2),
            'pct_revenue': round(group['Monetary'].sum() / rfm['Monetary'].sum() * 100, 1),
        }
        for seg, group in rfm.groupby('Segment')
    },
}

with open(OUTPUT_DIR / 'rfm_summary.json', 'w') as f:
    json.dump(rfm_summary, f, indent=2)

print(f'Saved summary: {OUTPUT_DIR / "rfm_summary.json"}')

---

## Summary

**Method:** Quintile-based RFM scoring with 11-segment standard framework.

**Key outputs:**
- Per-customer RFM table with R/F/M/FM scores and segment label.
- Segment economics table showing customer share vs revenue share.
- Saved artifacts: `rfm_scored.parquet`, `rfm_summary.json`.

**Validation:**
- Quintile bins are roughly balanced.
- Segment box plots show expected separation along R, F, M dimensions.
- Pareto pattern visible: top 20% customers ≈ 70-80% revenue.

**Limitations:**
- RFM treats all customers equally regardless of tenure. A customer who joined 1 month ago and bought once cannot be a Champion mathematically.
- Frequency does not account for category diversity (someone who buys 10 of the same product = same frequency as someone who buys 10 different products).
- Monetary uses raw revenue, not margin. High-revenue, low-margin customers may be over-prioritized.

**Next notebook:** `04_customer_clustering.ipynb` — apply K-Means and Hierarchical clustering to RFM features, compare with rule-based segments, identify any patterns RFM misses.

---

## Versioning

| Version | Date | Changes |
|---------|------|---------|
| **v1.0** | **April 26, 2026** | Initial RFM scoring with 11-segment framework, segment economics, and Pareto analysis. |